In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import PromptTemplate
# load all environment variables from .env
load_dotenv()
# Load groq API key into environment variable
os.environ["GROK_API_KEY"] = os.getenv("GROK_API_KEY")
# LangSmith Tracking configuration
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY_SUMMARIZATION_TEXT"] = os.getenv("LANGCHAIN_API_KEY_SUMMARIZATION_TEXT")
os.environ["LANGCHAIN_TRACKING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT_TEXT_SUMMARIZATION"] = os.getenv("LANGCHAIN_PROJECT_TEXT_SUMMARIZATION")
groq_api_key = os.getenv("GROK_API_KEY")

In [17]:
from langchain_core.documents import Document   # only needed for type hints
# ── 2. Helper function to combine documents ─────────────────────────────
def format_documents(docs: list[Document]) -> str:
    """
    Joins the page_content of multiple documents with double newlines.
    Used as the input formatter for the 'stuff' summarization pattern.
    """
    return "\n\n".join(doc.page_content for doc in docs)

def format_individual_doc(doc) -> str:
    """Format a single document for the map step."""
    return doc.page_content   # or doc.page_content.strip() if you want to clean

def format_all_summaries(summaries: list[str]) -> str:
    """Join intermediate summaries with double newlines for the reduce step."""
    separator = "\n" + "-" * 60 + "\n"
    return separator.join(summaries)

In [ ]:
llm_model = ChatGroq(model="llama-3.3-70b-versatile", groq_api_key=groq_api_key)

In [2]:
# Load and process documents
loader = PyPDFLoader("autosar_can_interface.pdf")
documents = loader.load_and_split()
documents

[Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-11-24T17:09:14+01:00', 'author': 'AUTOSAR', 'title': 'Specification of CAN Interface', 'subject': 'AUTOSAR', 'keywords': 'Release R25-11', 'moddate': '2025-11-24T17:09:14+01:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'source': 'autosar_can_interface.pdf', 'total_pages': 245, 'page': 0, 'page_label': '1'}, page_content='Specification of CAN Interface\nAUTOSAR CP R25-11\nDocument Title Specification of CAN Interface\nDocument Owner AUTOSAR\nDocument Responsibility AUTOSAR\nDocument Identification No 12\nDocument Status published\nPart of AUTOSAR Standard Classic Platform\nPart of Standard Release R25-11\nDocument Change History\nDate Release Changed by Description\n2025-11-27 R25-11\nAUTOSAR\nRelease\nManagement\n• Abstraction from Driver APIs\n• Removal of TTCan\n• Improve mask based receptio

In [ ]:
generic_template = """ 
You are summarization assistant. Your task is to summarize the content of the document in a concise and clear manner.
document: {document}
"""
prompt = PromptTemplate.from_template(generic_template)

## Stuff Chain Document Summerization

In [16]:

from langchain_core.output_parsers import StrOutputParser
# ── 3. Create the LCEL chain ────────────────────────────────────────────
# stuff_chain = (
#     # Step 1: documents list → single string
#     {"document": format_documents}
#     # Step 2: string → prompt (PromptValue)
#     | prompt
#     # Step 3: prompt → LLM call → AIMessage / str
#     | llm_model
#     # Step 4: AIMessage → clean string
#     | StrOutputParser()
# )
# output_summary = stuff_chain.invoke(documents)
# output_summary

## Map Reduce to summarize large documents

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 2000, chunk_overlap = 100)
final_documents = text_splitter.split_documents(documents)
len(final_documents)

329

In [18]:
# ── Your prompts (slightly cleaned up) ────────────────────────────────────────
chunks_prompt = """\
You are a summarization assistant. Your task is to summarize the content of the speech/document in a concise and clear manner.

speech: {document}
"""

final_prompt = """\
Provide the final summary of the document with the following points in mind:
- Add motivating and engaging language to make the summary more compelling and interesting to read.
- Use a clear and concise writing style to ensure that the summary is easy to understand and engaging for the reader.

Here are the extracted summaries from different parts of the speech/document:

{doc_summaries}

Final summary:"""

map_prompt   = PromptTemplate.from_template(chunks_prompt)
reduce_prompt = PromptTemplate.from_template(final_prompt)

In [19]:
from langchain_core.runnables import RunnablePassthrough

# ── Map step: summarize each chunk independently ──────────────────────────────
map_chain = (
    {"document": format_individual_doc}           # or RunnableLambda(lambda doc: doc.page_content)
    | map_prompt
    | llm_model                              # ← your LLM / ChatModel here
    | StrOutputParser()
)

# ── Reduce step: combine all chunk summaries into final summary ───────────────
reduce_chain = (
    {"doc_summaries": format_all_summaries}
    | reduce_prompt
    | llm_model
    | StrOutputParser()
)

# ── Full map-reduce chain ─────────────────────────────────────────────────────
# .map() applies map_chain to every document in the list in parallel
map_reduce_chain = map_chain.map() | reduce_chain

# ── Optional: also get intermediate chunk summaries ───────────────────────────
map_reduce_with_steps = map_chain.map().with_config({"run_name": "Map step"}) | {
    "intermediate_summaries": RunnablePassthrough(),
    "final_summary": reduce_chain
}

In [21]:
# ── Usage examples ────────────────────────────────────────────────────────────

# documents = [Document(page_content="chunk 1 ..."), Document(...), ...]

# 1. Just the final summary (most common)
# final_summary = map_reduce_chain.invoke(documents)

# # 2. Get both intermediate summaries + final one
# result = map_reduce_with_steps.invoke(documents)
# print("Intermediate summaries:")
# for i, s in enumerate(result["intermediate_summaries"], 1):
#     print(f"\nChunk {i}:\n{s}")
# print("\nFinal summary:\n", result["final_summary"])